# MedRAG — Medical Retrieval-Augmented Generation

**Student:** GACEMI RAIDA  
**Project type:** Retrieval-Augmented Generation (RAG)  
**Main language:** Python  
**Environment:** Google Colab

## 1. Introduction

MedRAG is a medical question-answering prototype that combines semantic document retrieval with large language model (LLM) generation. Its purpose is to retrieve relevant passages from a selected collection of medical PDF documents and provide those passages to an LLM, which then produces a natural-language response grounded in the retrieved context.

The project follows the two main stages of a RAG system:

1. **Retrieval:** extract text from medical PDFs, divide it into smaller chunks, convert each chunk into an embedding, and use FAISS to retrieve passages that are semantically similar to a user's question.
2. **Generation:** provide the retrieved passages, their document names and page numbers, and the user's question to Qwen2.5-3B-Instruct. Qwen generates an answer based on the supplied context.

The notebook builds these stages step by step and tests the retrieval and generation components.

## 2. Project objectives

- Build a searchable corpus from medical PDF documents.
- Preserve the source filename and page number for each extracted passage.
- Generate multilingual semantic embeddings for the document chunks.
- Build a FAISS vector index for similarity search.
- Retrieve relevant passages in response to natural-language questions.
- Use Qwen2.5-3B-Instruct to generate answers from the retrieved context.
- Examine the limitations of retrieval and generated answers.

## 3. System architecture

```text
Medical PDF documents
        |
        v
Text extraction (PyMuPDF)
        |
        v
Medical document selection
        |
        v
Text chunking (900 characters, 150 overlap)
        |
        v
Embeddings (multilingual MiniLM, 384 dimensions)
        |
        v
FAISS vector index
        |
User question
        |
        v
Question embedding + FAISS similarity search
        |
        v
Retrieved passages + source/page metadata
        |
        v
Prompt construction
        |
        v
Qwen2.5-3B-Instruct
        |
        v
Generated answer
```

## 4. Technologies

| Technology | Role in this project |
|---|---|
| PyMuPDF | Extract selectable text from PDF pages |
| LangChain text splitters | Divide page text into overlapping chunks |
| Sentence Transformers | Load the embedding model and encode text |
| `paraphrase-multilingual-MiniLM-L12-v2` | Convert questions and chunks into 384-dimensional embeddings |
| FAISS | Store and search document vectors |
| Hugging Face Transformers | Load the tokenizer and Qwen language model |
| Qwen2.5-3B-Instruct | Generate answers from the retrieved context |
| PyTorch | Run tensor operations and model inference |

## 5. Corpus and scope

The notebook discovers PDF files in a Google Drive folder. It extracts selectable text and preserves the source filename and page number.

The current medical corpus excludes `State-of-wildfires-2024-2025.pdf`. The medical collection therefore consists of the three medical PDFs selected in the notebook. OCR is not implemented: pages without extractable text are skipped.

The notebook's recorded run extracted **1,584 medical pages** and produced **6,615 text chunks**. These figures describe the run represented by the saved notebook outputs; they may change if the input PDFs or extraction process change.

## 6. Important note

This is an educational research prototype, not a clinically validated medical decision-support system. Retrieved passages may be incomplete or irrelevant, and an LLM may misinterpret or overgeneralize them. Answers must be checked against the original documents and qualified medical sources. The system must not be used to diagnose, prescribe, or replace professional medical judgment.

---

In [4]:

!pip -q install pymupdf langchain-text-splitters sentence-transformers faiss-cpu transformers accelerate


## Stage 1 — Install dependencies

The first cell installs the Python libraries used throughout the notebook:

- `pymupdf` for reading PDF files and extracting text.
- `langchain-text-splitters` for dividing long text into chunks.
- `sentence-transformers` for loading and using the embedding model.
- `faiss-cpu` for vector similarity search.
- `transformers` and `accelerate` for loading and running Qwen.

Run this cell once at the beginning of a fresh Colab runtime. If the runtime is reset, the libraries may need to be installed again.

Import libraries and configure the environment

In [3]:

import json
import random
from pathlib import Path

import faiss
import numpy as np
import pymupdf
import torch

from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

# Reproducibility
random.seed(42)
np.random.seed(42)

# Check GPU availability
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


GPU available: True
GPU: Tesla T4


## Stage 2 — Connect Google Drive

The source PDFs are stored in Google Drive rather than inside the notebook. Mounting Drive allows Colab to read those files.

When running this notebook, sign in to the Google account that contains the project data and authorize the Drive connection.

Access Google Drive and discover medical PDFs

In [5]:

from google.colab import drive

drive.mount('/content/drive')


Mounted at /content/drive


### Configure the data folders

`PDF_FOLDER` points to the folder containing the source PDFs. `DATA_DIR` is used for generated index-related data and is created if it does not already exist.

**Before running:** make sure the folder path matches the location of your own files in Google Drive. The current path is specific to the original Colab setup.

In [6]:

from pathlib import Path

# Folder containing your medical PDFs
PDF_FOLDER = Path(
    "/content/drive/MyDrive/MedRAG/medrag_data"
)

# Folder for generated data
DATA_DIR = Path(
    "/content/drive/MyDrive/MedRAG/medrag_data/index_data"
)

# Create the output folder if it doesn't exist
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("PDF folder:", PDF_FOLDER)
print("Output folder:", DATA_DIR)
print("PDF folder exists:", PDF_FOLDER.exists())


PDF folder: /content/drive/MyDrive/MedRAG/medrag_data
Output folder: /content/drive/MyDrive/MedRAG/medrag_data/index_data
PDF folder exists: True


In [7]:

# Find all PDFs, including those in subfolders
pdf_paths = sorted(
    [
        p for p in PDF_FOLDER.rglob("*")
        if p.is_file()
        and p.suffix.lower() == ".pdf"
        and DATA_DIR not in p.parents
    ],
    key=lambda p: str(p).lower()
)

print(f"{len(pdf_paths)} PDF(s) found:\n")

for p in pdf_paths:
    print("📄", p.name)

if not pdf_paths:
    raise FileNotFoundError(
        "No PDFs found. Check the Google Drive folder path."
    )


4 PDF(s) found:

📄 40mbs_medicalbooksstore_2017_essentials.pdf
📄 8205Oxford Handbook of Clinical Medicine 10th 2017 Edition_SamanSarKo - Copy.pdf
📄 clinical_medicine_ashok_chandra.pdf
📄 State-of-wildfires-2024-2025.pdf


## Stage 3 — Inspect the PDF collection

Before extracting text, the notebook discovers PDF files and inspects their filenames, page counts, file sizes and encryption status.

This inspection helps verify that the expected files are available and readable. The next diagnostic cell previews selectable text from the first pages of each PDF.

understand documents


In [8]:

import fitz
import pandas as pd

pdf_info = []

for pdf_path in pdf_paths:
    doc = fitz.open(pdf_path)

    pdf_info.append({
        "Filename": pdf_path.name,
        "Pages": len(doc),
        "Size (MB)": round(pdf_path.stat().st_size / (1024 * 1024), 2),
        "Encrypted": doc.is_encrypted
    })

    doc.close()

pdf_info_df = pd.DataFrame(pdf_info)

display(pdf_info_df)


/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


,Filename,Pages,Size (MB),Encrypted
0,40mbs_medicalbooksstore_2017_essentials.pdf,638,12.51,False
1,8205Oxford Handbook of Clinical Medicine 10th ...,911,31.47,False
2,clinical_medicine_ashok_chandra.pdf,52,0.58,False
3,State-of-wildfires-2024-2025.pdf,112,13.13,False


In [9]:

# ==========================================
# PDF TEXT EXTRACTION TEST
# ==========================================

for pdf_path in pdf_paths:

    doc = fitz.open(pdf_path)

    print("\n" + "=" * 80)
    print("DOCUMENT:", pdf_path.name)
    print("=" * 80)

    # Inspect the first 3 pages
    for page_num in range(min(3, len(doc))):

        page = doc[page_num]
        text = page.get_text("text")

        print(f"\n--- PAGE {page_num + 1} ---")
        print("Extracted characters:", len(text))
        print("Text preview:")
        print(text[:500] if text.strip() else "[NO TEXT EXTRACTED]")

    doc.close()



DOCUMENT: 40mbs_medicalbooksstore_2017_essentials.pdf

--- PAGE 1 ---
Extracted characters: 84
Text preview:
h Edition 
Ardhendu Sinha Ray 
Abhise 
Si a a.v 
https://t.me/MBS_MedicalBooksStore


--- PAGE 2 ---
Extracted characters: 34
Text preview:
Essentials of  
INTERNAL MEDICINE


--- PAGE 3 ---
Extracted characters: 569
Text preview:
Essentials of  
INTERNAL MEDICINE
Ardhendu Sinha Ray MD
Ex-Professor 
Department of Medicine
Kolkata Medical College
RG Kar Medical College
Kolkata, West Bengal, India
Burdwan Medical College
Burdwan, West Bengal, India
Kalinga Institute of Medical Sciences 
Bhubaneswar, Odisha, India
Abhisekh Sinha Ray MD (NY)
Staff Nephrologist
Department of Nephrology and Hypertension
Good Samaritan Medical Center 
NE, USA
Formerly, Fellow
Kidney Institute, University of Kansas Medical Center
Kansas, USA
New 

DOCUMENT: 8205Oxford Handbook of Clinical Medicine 10th 2017 Edition_SamanSarKo - Copy.pdf

--- PAGE 1 ---
Extracted characters: 0
Text preview:
[NO TEXT EXT

### Extract selectable text and preserve metadata

PyMuPDF opens each PDF and reads the text available on every page. Empty pages are skipped.

Each extracted page is stored as a dictionary with three fields:

- `text`: the extracted page text.
- `source`: the original PDF filename.
- `page`: the page number, starting at 1.

Keeping the source and page metadata makes it possible to trace retrieved passages back to their original documents.

**Scope limitation:** this notebook does not perform OCR. Text embedded only in scanned images will not be extracted by this method.

In [10]:

import fitz

documents = []

for pdf_path in pdf_paths:

    doc = fitz.open(pdf_path)

    for page_num, page in enumerate(doc):

        text = page.get_text("text").strip()

        if not text:
            continue

        documents.append({
            "text": text,
            "source": pdf_path.name,
            "page": page_num + 1
        })

    doc.close()

print("Extraction completed!")
print("Total extracted pages:", len(documents))


Extraction completed!
Total extracted pages: 1696


In [11]:

from collections import Counter

pages_per_document = Counter(
    item["source"] for item in documents
)

for filename, count in pages_per_document.items():
    print(f"{filename}: {count} extracted pages")

print("\nTotal extracted pages:", len(documents))


40mbs_medicalbooksstore_2017_essentials.pdf: 625 extracted pages
8205Oxford Handbook of Clinical Medicine 10th 2017 Edition_SamanSarKo - Copy.pdf: 909 extracted pages
clinical_medicine_ashok_chandra.pdf: 50 extracted pages
State-of-wildfires-2024-2025.pdf: 112 extracted pages

Total extracted pages: 1696


### Select the medical corpus

The notebook initially discovers all PDFs in the configured folder. This filtering step creates `medical_documents` by excluding `State-of-wildfires-2024-2025.pdf`.

Only the three medical PDFs are used for chunking, embedding and retrieval. The assertion checks that the wildfire report is not present in the selected medical corpus.

In [12]:

# ==========================================
# STEP 6.5: SELECT MEDICAL DOCUMENTS ONLY
# ==========================================

medical_documents = [
    doc for doc in documents
    if doc["source"] != "State-of-wildfires-2024-2025.pdf"
]

print("Medical corpus created successfully!")
print("Total medical pages:", len(medical_documents))

# Verify the selected documents
from collections import Counter

medical_stats = Counter(
    doc["source"] for doc in medical_documents
)

for filename, count in medical_stats.items():
    print(f"{filename}: {count} pages")

# Verify that wildfire is excluded
assert all(
    doc["source"] != "State-of-wildfires-2024-2025.pdf"
    for doc in medical_documents
)

print("\nWildfire document excluded successfully!")


Medical corpus created successfully!
Total medical pages: 1584
40mbs_medicalbooksstore_2017_essentials.pdf: 625 pages
8205Oxford Handbook of Clinical Medicine 10th 2017 Edition_SamanSarKo - Copy.pdf: 909 pages
clinical_medicine_ashok_chandra.pdf: 50 pages

Wildfire document excluded successfully!


## Stage 4 — Split the extracted text into chunks

RAG systems usually retrieve smaller passages rather than entire books or full PDF pages. Chunking divides long text into manageable pieces that can be embedded and searched.

This notebook uses LangChain's `RecursiveCharacterTextSplitter`. It attempts to split at paragraph and line boundaries before falling back to spaces or individual characters.

 Text chunking with LangChain

In [13]:

from langchain_text_splitters import RecursiveCharacterTextSplitter


### Chunking configuration

- **Chunk size: 900 characters.** This sets the maximum target length of each chunk.
- **Chunk overlap: 150 characters.** Adjacent chunks share some text so that information near a boundary is less likely to be lost.

The overlap can preserve context when a sentence or explanation is close to a chunk boundary. Each chunk retains the source filename and page number of the page from which it was created.

Configure the splitter

In [14]:

# ==========================================
# STEP 7: CHUNKING CONFIGURATION
# ==========================================

CHUNK_SIZE = 900
CHUNK_OVERLAP = 150

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", " ", ""]
)

print("Text splitter configured!")
print("Chunk size:", CHUNK_SIZE)
print("Chunk overlap:", CHUNK_OVERLAP)


Text splitter configured!
Chunk size: 900
Chunk overlap: 150


Generate chunks from medical documents

In [15]:

# ==========================================
#  GENERATE MEDICAL CHUNKS
# ==========================================

chunks = []

for document in medical_documents:

    page_chunks = text_splitter.split_text(
        document["text"]
    )

    for chunk in page_chunks:

        chunks.append({
            "text": chunk,
            "source": document["source"],
            "page": document["page"]
        })

print("Chunking completed!")
print("Total chunks:", len(chunks))


Chunking completed!
Total chunks: 6615


### Inspect and validate the chunks

The following cells display an example chunk and calculate basic statistics such as average, minimum and maximum length.

These checks help verify that chunking worked and that no chunk exceeds the configured maximum size. A short or noisy chunk may still occur, so chunk statistics do not by themselves guarantee retrieval quality.

Inspect  generated chunks

In [16]:

# Display the first chunk

print("SOURCE:", chunks[0]["source"])
print("PAGE:", chunks[0]["page"])
print("CHARACTERS:", len(chunks[0]["text"]))

print("\nCHUNK CONTENT:\n")
print(chunks[0]["text"])


SOURCE: 40mbs_medicalbooksstore_2017_essentials.pdf
PAGE: 1
CHARACTERS: 83

CHUNK CONTENT:

h Edition 
Ardhendu Sinha Ray 
Abhise 
Si a a.v 
https://t.me/MBS_MedicalBooksStore


In [17]:

chunk_lengths = [
    len(chunk["text"])
    for chunk in chunks
]

print("Total chunks:", len(chunks))
print("Average chunk length:", round(np.mean(chunk_lengths), 2))
print("Minimum chunk length:", min(chunk_lengths))
print("Maximum chunk length:", max(chunk_lengths))

print("Chunks exceeding 900 characters:",
      sum(length > CHUNK_SIZE for length in chunk_lengths))


Total chunks: 6615
Average chunk length: 782.56
Minimum chunk length: 11
Maximum chunk length: 900
Chunks exceeding 900 characters: 0


## Stage 5 — Generate semantic embeddings

An embedding is a numerical vector representing text. Texts with related meanings can have vectors that are close to one another in the embedding space.

The notebook uses Sentence Transformers with:

`sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`

The model produces **384-dimensional** embeddings. It is used for both document chunks and user questions so that they can be compared in the same vector space.

Embedding generation

In [18]:

from sentence_transformers import SentenceTransformer
import torch

# Select device
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Selected device:", device)


Selected device: cuda


In [19]:

# ==========================================
#  LOAD EMBEDDING MODEL
# ==========================================

EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL,
    device=device
)

print("Embedding model loaded successfully!")
print("Model:", EMBEDDING_MODEL)
print("Embedding dimension:",
      embedding_model.get_sentence_embedding_dimension())


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!
Model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Embedding dimension: 384


/tmp/ipykernel_314/2284325494.py:15: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_model.get_sentence_embedding_dimension())


In [20]:

# Two semantically similar sentences
sentences = [
    "Diabetes is characterized by high blood glucose levels.",
    "Hyperglycemia is an elevated level of glucose in the blood."
]

test_embeddings = embedding_model.encode(
    sentences,
    convert_to_numpy=True
)

print("Embedding shape:", test_embeddings.shape)
print("\nFirst sentence vector:")
print(test_embeddings[0][:10])

print("\nSecond sentence vector:")
print(test_embeddings[1][:10])


Embedding shape: (2, 384)

First sentence vector:
[-0.26375085  0.34312057 -0.22777118  0.4612379  -0.06767667 -0.29221052
  0.30315727 -0.01394811  0.08837619 -0.05517971]

Second sentence vector:
[-0.17179982  0.26449093 -0.21027128  0.4287644   0.07369567 -0.25292766
  0.31253368  0.11796328  0.04541186  0.10295352]


In [21]:

from sentence_transformers import util

similarity = util.cos_sim(
    test_embeddings[0],
    test_embeddings[1]
)

print("Cosine similarity:", similarity.item())


Cosine similarity: 0.8025462031364441


In [22]:

# ==========================================
# STEP 8.6: GENERATE ALL EMBEDDINGS
# ==========================================

chunk_texts = [
    chunk["text"] for chunk in chunks
]

embeddings = embedding_model.encode(
    chunk_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embedding generation completed!")
print("Embedding matrix shape:", embeddings.shape)
print("Data type:", embeddings.dtype)


Batches:   0%|          | 0/207 [00:00<?, ?it/s]

Embedding generation completed!
Embedding matrix shape: (6615, 384)
Data type: float32


### Validate the embeddings

The validation cell checks the embedding matrix shape, data type, missing values, infinite values and vector norms.

Embeddings are generated with `normalize_embeddings=True`. Normalized vectors have a norm close to 1, which is important because the FAISS index below uses inner-product similarity. For normalized vectors, inner product is equivalent to cosine similarity.

Validate embeddings

In [23]:

# ==========================================
# STEP 8.7: EMBEDDING VALIDATION
# ==========================================

print("Shape:", embeddings.shape)
print("Data type:", embeddings.dtype)

# Check missing or infinite values
print("Contains NaN:", np.isnan(embeddings).any())
print("Contains Inf:", np.isinf(embeddings).any())

# Check vector norms
norms = np.linalg.norm(embeddings, axis=1)

print("Minimum norm:", norms.min())
print("Maximum norm:", norms.max())
print("Average norm:", norms.mean())


Shape: (6615, 384)
Data type: float32
Contains NaN: False
Contains Inf: False
Minimum norm: 0.9999999
Maximum norm: 1.0000001
Average norm: 1.0


## Stage 6 — Build the FAISS vector index

FAISS is a library for efficient similarity search over vectors. It stores the chunk embeddings and returns the indices of the closest vectors to a query embedding.

This notebook uses `IndexFlatIP`, an exact inner-product index. It compares the query vector against the stored vectors directly. Because the vectors are normalized, the returned inner-product scores correspond to cosine similarity.

FAISS stores the vectors, not the original text. The returned vector indices are used to look up the matching entries in the `chunks` list, where the text and source metadata are stored.

Build the FAISS vector index

In [24]:

import faiss
import numpy as np


In [25]:

# ==========================================
# CREATE FAISS INDEX
# ==========================================

# Ensure embeddings have the correct format
embeddings = np.ascontiguousarray(
    embeddings.astype("float32")
)

# Get embedding dimension
dimension = embeddings.shape[1]

# Create exact inner-product index
faiss_index = faiss.IndexFlatIP(dimension)

# Add all medical embeddings
faiss_index.add(embeddings)

print("FAISS index created successfully!")
print("Index dimension:", faiss_index.d)
print("Number of indexed vectors:", faiss_index.ntotal)


FAISS index created successfully!
Index dimension: 384
Number of indexed vectors: 6615


## Stage 7 — Retrieve relevant passages

When a user asks a question, `retrieve_documents()`:

1. Encodes the question with the same MiniLM embedding model.
2. Converts the query vector to the `float32` format expected by FAISS.
3. Searches the FAISS index for the `top_k` closest vectors.
4. Uses each returned index to retrieve the corresponding chunk text, source filename and page number.
5. Returns the passages together with their similarity scores.

The default is `top_k=3`. A higher similarity score indicates a closer vector match, not proof that a passage directly answers the question. Retrieval quality must therefore be evaluated using the actual content of the returned passages.

Semantic retrieval

In [26]:

def retrieve_documents(query, top_k=3):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    query_embedding = np.ascontiguousarray(
        query_embedding.astype("float32")
    )

    scores, indices = faiss_index.search(
        query_embedding,
        top_k
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):

        if idx == -1:
            continue

        results.append({
            "text": chunks[idx]["text"],
            "source": chunks[idx]["source"],
            "page": chunks[idx]["page"],
            "score": float(score)
        })

    return results


### Retrieval experiments

The next cells test the retriever with medical questions about diabetes, hypertension, acute kidney injury and pneumonia.

Inspect the passage text as well as the similarity score. A passage can mention the same medical term without answering the exact question. For example, a passage about diabetic ketoacidosis is not automatically a complete answer about general diabetes symptoms.

In [27]:
query = "What are the common symptoms of diabetes?"

results = retrieve_documents(query)

for i, result in enumerate(results, 1):
    print(f"\nRESULT {i}")
    print("Score:", result["score"])
    print("Source:", result["source"])
    print("Page:", result["page"])
    print("Text:", result["text"])


RESULT 1
Score: 0.7199085354804993
Source: 40mbs_medicalbooksstore_2017_essentials.pdf
Page: 434
Text: ••
Occurrence of diabetic neuropathy correlates with:
−−Duration of DM
−−Adequacy of glycemic control.
Diabetic neuropathy are of four types:
••
Polyneuropathy (most common)
••
Mononeuropathy/mononeuropathy multiplex
••
Radiculopathy
••
Autonomic neuropathy.
Polyneuropathy
••
It is the most common form of diabetic neuropathy.
••
It frequently presents with distal sensory loss/
paresthesia and dysesthesia.
••
Symptoms of polyneuropathy also include numbness, 
tingling, sharpness and burning that begin at the feet 
and spread proximally.
••
Neuropathic pain developed in some individual 
preceded by improvement in glycemic control which is 
usually present at rest and worsen at night mainly found 
in lower extremities. This neuropathic pain may be of 
two types—acute or chronic. As neuropathy progresses 
the pain subsides but the sensory deficit increases.
••

RESULT 2
Score: 0.69308775

In [28]:

test_queries = [
    "What are the symptoms of hypertension?",
    "What causes acute kidney injury?",
    "What are the complications of diabetes?",
    "How is pneumonia diagnosed?"
]

for query in test_queries:

    print("\n" + "=" * 80)
    print("QUERY:", query)

    results = retrieve_documents(query, top_k=3)

    for i, result in enumerate(results, 1):

        print(f"\nResult {i}")
        print("Score:", round(result["score"], 4))
        print("Source:", result["source"])
        print("Page:", result["page"])
        print("Text:", result["text"][:350])



QUERY: What are the symptoms of hypertension?

Result 1
Score: 0.7149
Source: 40mbs_medicalbooksstore_2017_essentials.pdf
Page: 347
Text: Clinical Features
Common among adolescent and young ladies.
General symptoms are fever, malaise, night sweats and 
arthralgias.
Hypertension is seen up to 90% of patients and 
responsible for renal, cardiac and cerebral injury.
Vascular occlusion is manifested as
a.	 Involvement of subclavian (90%) manifested as 
Raynaud’s phenomenon, claudication 

Result 2
Score: 0.7054
Source: 40mbs_medicalbooksstore_2017_essentials.pdf
Page: 416
Text: 411
Adrenal Gland Disorders
Majority (90%) of the patients have hypertension that 
may be mild to severe, labile and difficult to treat.
Patient may present with cardiovascular collapse arrhy- 
thmia during operation under GA or following drugs like 
opiates, glucagon, metaclopramide, pancuronium and 
TCAD.
A detailed family history should be taken

Result 3
Score: 0.6889
Source: 8205Oxford Handbook of Clinical Med

In [29]:
test_query = "What are the common symptoms of hypertension?"

results = retrieve_documents(test_query, top_k=5)

for i, result in enumerate(results, 1):
    print("\n" + "=" * 90)
    print(f"RESULT {i}")
    print("Similarity:", round(result["score"], 4))
    print("Source:", result["source"])
    print("Page:", result["page"])
    print("\nText:\n", result["text"])


RESULT 1
Similarity: 0.7378
Source: 40mbs_medicalbooksstore_2017_essentials.pdf
Page: 347

Text:
 Clinical Features
Common among adolescent and young ladies.
General symptoms are fever, malaise, night sweats and 
arthralgias.
Hypertension is seen up to 90% of patients and 
responsible for renal, cardiac and cerebral injury.
Vascular occlusion is manifested as
a.	 Involvement of subclavian (90%) manifested as 
Raynaud’s phenomenon, claudication with unequal 
or absent radial pulse.
b.	 Involvement of common carotid (60%) manifested 
as syncope, TIA, CVA and visual change.
c.	 Involvement of renal artery (40%) manifested as 
hypertension and renal failure.
d.	 Occlusion of coronary artery (10%) manifested as 
AMI and angina.
e.	 Involvement of  aortic root and arch (35%) causes AR 
and CCF.
Laboratory  Features
••
Nonspecific—Anemia, raised ESR and with elevated 
immunoglobulin.
Diagnosis
Young ladies with decrease or absent peripheral pulse,

RESULT 2
Similarity: 0.7255
Source: 40mbs_m

In [30]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(
        torch.cuda.get_device_properties(0).total_memory / (1024**3), 2
    ), "GB")
else:
    print("No GPU detected. Current device: CPU")

CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


## Stage 8 — Load the Qwen language model

The retrieval component finds relevant passages, but it does not compose them into a natural-language answer. The generation component is provided by **Qwen2.5-3B-Instruct**, loaded through Hugging Face Transformers.

The tokenizer converts the prompt into token IDs that the model can process. The causal language model generates the response token by token.

The notebook uses FP16 precision and `device_map="auto"` to load the model on the available hardware. A GPU is recommended; the recorded run used an NVIDIA Tesla T4.

Load Qwen2.5-3B-Instruct

In [31]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load the language model
llm = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Set model to evaluation mode
llm.eval()

print("Qwen loaded successfully!")
print("Model:", MODEL_NAME)
print("Device:", llm.device)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


... (2 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen loaded successfully!
Model: Qwen/Qwen2.5-3B-Instruct
Device: cuda:0


## Stage 9 — Build the context for generation

`build_context()` converts the list of retrieved document dictionaries into one formatted text block.

For each passage, the context includes:
- A passage number.
- The source PDF filename.
- The original page number.
- The passage text.

This gives Qwen the retrieved evidence and preserves source information that can be used when presenting references.

Build the context from retrieved passages

In [32]:
def build_context(retrieved_documents):

    context = ""

    for i, document in enumerate(retrieved_documents, 1):

        context += f"""
[Passage {i}]
Source: {document['source']}
Page: {document['page']}

Content:
{document['text']}

"""

    return context

In [33]:
query = "What are the common symptoms of diabetes?"

retrieved_documents = retrieve_documents(query, top_k=3)

context = build_context(retrieved_documents)

print(context)


[Passage 1]
Source: 40mbs_medicalbooksstore_2017_essentials.pdf
Page: 434

Content:
••
Occurrence of diabetic neuropathy correlates with:
−−Duration of DM
−−Adequacy of glycemic control.
Diabetic neuropathy are of four types:
••
Polyneuropathy (most common)
••
Mononeuropathy/mononeuropathy multiplex
••
Radiculopathy
••
Autonomic neuropathy.
Polyneuropathy
••
It is the most common form of diabetic neuropathy.
••
It frequently presents with distal sensory loss/
paresthesia and dysesthesia.
••
Symptoms of polyneuropathy also include numbness, 
tingling, sharpness and burning that begin at the feet 
and spread proximally.
••
Neuropathic pain developed in some individual 
preceded by improvement in glycemic control which is 
usually present at rest and worsen at night mainly found 
in lower extremities. This neuropathic pain may be of 
two types—acute or chronic. As neuropathy progresses 
the pain subsides but the sensory deficit increases.
••


[Passage 2]
Source: 40mbs_medicalbooksstore_

## Stage 10 — Construct the RAG prompt

`build_prompt()` creates a chat conversation with two roles:

- **System message:** tells Qwen to answer from the supplied medical context, avoid unsupported claims, acknowledge insufficient information, distinguish general conditions from specific complications, and include document references.
- **User message:** contains the retrieved context and the original question.

Prompt instructions guide the model, but they do not guarantee that every answer will be correct or that citations will always be included. These properties must be evaluated.

Define the RAG prompt

In [34]:
def build_prompt(query, context):

    messages = [
        {
            "role": "system",
            "content": """
You are a medical information assistant.

Your task is to answer the user's question using only
the medical information provided in the context.

Instructions:
1. Use the provided context as your primary source.
2. Do not invent medical facts or unsupported information.
3. If the context does not contain enough information,
   clearly state that the available documents are insufficient.
4. Distinguish between general medical conditions
   and specific complications or subtypes.
5. Organize the answer clearly and concisely.
6. Include the source PDF filename and page number
   for the information used.
7. Do not provide a diagnosis or replace professional
   medical judgment.
"""
        },
        {
            "role": "user",
            "content": f"""
Medical context:

{context}

Question:
{query}

Please provide an answer based on the context.
"""
        }
    ]

    return messages

In [35]:
query = "What are the common symptoms of diabetes?"

retrieved_documents = retrieve_documents(query, top_k=3)

context = build_context(retrieved_documents)

messages = build_prompt(query, context)

for message in messages:
    print("\nROLE:", message["role"])
    print(message["content"])


ROLE: system

You are a medical information assistant.

Your task is to answer the user's question using only
the medical information provided in the context.

Instructions:
1. Use the provided context as your primary source.
2. Do not invent medical facts or unsupported information.
3. If the context does not contain enough information,
   clearly state that the available documents are insufficient.
4. Distinguish between general medical conditions
   and specific complications or subtypes.
5. Organize the answer clearly and concisely.
6. Include the source PDF filename and page number
   for the information used.
7. Do not provide a diagnosis or replace professional
   medical judgment.


ROLE: user

Medical context:


[Passage 1]
Source: 40mbs_medicalbooksstore_2017_essentials.pdf
Page: 434

Content:
••
Occurrence of diabetic neuropathy correlates with:
−−Duration of DM
−−Adequacy of glycemic control.
Diabetic neuropathy are of four types:
••
Polyneuropathy (most common)
••
Mononeu

## Stage 11 — Tokenize the prompt and generate an answer

The chat template formats the system and user messages according to Qwen's expected conversation format. The tokenizer converts this formatted input into tensors, which are moved to the same device as the model.

The `generate()` method then produces new tokens. The notebook removes the input portion of the output and decodes only the newly generated tokens into readable text.

`max_new_tokens` limits the response length. `do_sample=False` uses greedy decoding, selecting the highest-probability next token at each generation step.

In [36]:
# Convert messages into Qwen's chat format
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True
)

# Move all input tensors to the GPU
inputs = {
    key: value.to(llm.device)
    for key, value in inputs.items()
}

print("Input shape:", inputs["input_ids"].shape)
print("Input device:", inputs["input_ids"].device)

Input shape: torch.Size([1, 826])
Input device: cuda:0


In [37]:
with torch.inference_mode():

    outputs = llm.generate(
        **inputs,
        max_new_tokens=400,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

# Extract only newly generated tokens
generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

# Decode the answer
answer = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print("MEDRAG ANSWER")
print("=" * 80)
print(answer)

MEDRAG ANSWER
Based on the provided medical context, the common symptoms of diabetes include:

- Unexplained vomiting
- Abdominal pain
- Polyuria (frequent urination)
- Polydipsia (increased thirst)
- Lethargy
- Anorexia (loss of appetite)
- Ketotic breath
- Dehydration
- Coma

These symptoms can indicate diabetic ketoacidosis (DKA), which is a severe complication of poorly managed diabetes. However, it's important to note that these symptoms are not specific to diabetes and could be indicative of other health issues as well.


## 12. Results, limitations and future work

### Current implementation

The recorded notebook run completed the main RAG pipeline:

- 1,584 extracted pages in the selected medical corpus.
- 6,615 text chunks.
- 384-dimensional normalized embeddings.
- A FAISS exact inner-product index.
- Semantic retrieval of medical passages.
- Qwen2.5-3B-Instruct answer generation on a Tesla T4 GPU.

### Observed limitations

1. **Retrieval relevance:** the most similar passage is not always the passage that best answers the question.
2. **Context completeness:** the answer can only be as useful as the passages retrieved and supplied to the model.
3. **Medical overgeneralization:** Qwen may describe symptoms of a specific complication as if they were general symptoms of a disease.
4. **Citation reliability:** including source information in the prompt does not guarantee that the generated answer will cite every source correctly.
5. **PDF text extraction:** scanned pages and image-only content are not processed because OCR is not implemented.
6. **No clinical validation:** the system has not been validated for clinical use.

### Possible future improvements

- Evaluate retrieval with a curated set of medical questions and relevant source passages.
- Improve chunk quality and filter irrelevant or noisy chunks.
- Experiment with a reranker to reorder retrieved candidates.
- Improve answer grounding and verify every generated citation against the retrieved passages.
- Add an evaluation framework for faithfulness, relevance and answer completeness.
- Consider OCR only if scanned documents become part of the intended corpus.

### Responsible-use statement

MedRAG is an educational prototype for exploring retrieval-augmented generation over medical documents. It is not a medical device and must not be used as a substitute for a qualified healthcare professional, diagnosis or treatment advice.